# Two-building workflow test

This notebook simulates the workflow you described:
1. Generate and modify building footprints in Python.
2. Place building A in Rhino or Grasshopper.
3. Ask a Rhino-side tool for remaining feasible positions for building B.
4. Check whether a user-requested point is geometrically possible.
5. Let the LLM combine those geometry facts with architectural intent before final placement.

## Planner note

Yes, this should become a planner case. The planner should eventually manage a repeated sub-sequence for each building:
`generate -> evaluate requested point -> analyze remaining positions -> place -> repeat or report`.

The geometry tools below are mocks so you can validate the workflow before the real Grasshopper tools exist.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from agent.tools import (
    generate_building_boundary,
    mock_check_requested_position,
    mock_import_building_boundary,
    mock_remaining_buildable_positions,
)

OUTPUT_DIR = Path.cwd()
BUILDING_A_JSON = OUTPUT_DIR / 'two_building_building_a_payload.json'
BUILDING_B_JSON = OUTPUT_DIR / 'two_building_building_b_payload.json'
REMAINING_JSON = OUTPUT_DIR / 'two_building_remaining_positions.json'

In [ ]:
site_boundary = [
    [0.0, 0.0, 0.0],
    [140.0, 0.0, 0.0],
    [140.0, 90.0, 0.0],
    [0.0, 90.0, 0.0],
    [0.0, 0.0, 0.0],
]

user_intent = {
    'building_a': 'Place the first building near the west side for a quieter frontage.',
    'building_b': 'The second building should try to face the courtyard, but the user may request a specific point.',
}

requested_point_for_building_b = [45.0, 45.0]
site_boundary

In [ ]:
building_a = generate_building_boundary(
    area=700.0,
    building_type='I',
    building_depth=14.0,
    location_xy=(35.0, 45.0),
)

building_b_prototype = generate_building_boundary(
    area=520.0,
    building_type='L',
    building_depth=14.0,
    shape_ratio=0.62,
)

building_a['data']['geometry_id'], building_b_prototype['data']['geometry_id']

In [ ]:
placed_a = mock_import_building_boundary(
    geometry_id=building_a['data']['geometry_id'],
    boundary=building_a['data']['boundary'],
)

placed_a

In [ ]:
remaining_positions = mock_remaining_buildable_positions(
    site_boundary=site_boundary,
    placed_buildings=[building_a['data']],
    candidate_building_boundary=building_b_prototype['data']['boundary'],
    grid_size=10.0,
    clearance=4.0,
    max_positions=20,
)

remaining_positions['data']['candidate_count'], remaining_positions['data']['candidate_positions'][:8]

In [ ]:
requested_position_check = mock_check_requested_position(
    site_boundary=site_boundary,
    placed_buildings=[building_a['data']],
    proposed_boundary=building_b_prototype['data']['boundary'],
    requested_point=requested_point_for_building_b,
    candidate_positions=remaining_positions['data']['candidate_positions'],
    clearance=4.0,
    max_suggestions=5,
)

requested_position_check

## How the LLM should use this

The geometry checker should not make the architectural decision by itself. It should only return facts such as:
- the point is feasible or not
- whether the footprint overlaps another building
- whether the footprint leaves the site
- the nearest feasible alternatives

Then the LLM combines those facts with the user narrative, for example:
- `Yes, that point works geometrically and still keeps the courtyard open.`
- `No, that point collides with building A, but these nearby options preserve the same frontage intent.`

In [ ]:
chosen_point_for_building_b = (
    requested_position_check['data']['requested_point'][:2]
    if requested_position_check['data']['is_feasible']
    else requested_position_check['data']['suggested_positions'][0][:2]
)

chosen_point_for_building_b

In [ ]:
building_b_final_position = mock_check_requested_position(
    site_boundary=site_boundary,
    placed_buildings=[building_a['data']],
    proposed_boundary=building_b_prototype['data']['boundary'],
    requested_point=list(chosen_point_for_building_b),
    candidate_positions=remaining_positions['data']['candidate_positions'],
    clearance=4.0,
    max_suggestions=3,
)

placed_b = mock_import_building_boundary(
    geometry_id=building_b_prototype['data']['geometry_id'],
    boundary=building_b_final_position['data']['translated_boundary'],
)

placed_b

In [ ]:
building_a_payload = {
    'geometry_id': building_a['data']['geometry_id'],
    'building_footprint': {
        'type': 'Polygon',
        'coordinates': building_a['data']['boundary'],
    },
    'placement': placed_a['data'],
}

building_b_payload = {
    'geometry_id': building_b_prototype['data']['geometry_id'],
    'building_footprint': {
        'type': 'Polygon',
        'coordinates': building_b_final_position['data']['translated_boundary'],
    },
    'placement': placed_b['data'],
    'requested_point_check': requested_position_check['data'],
}

BUILDING_A_JSON.write_text(json.dumps(building_a_payload, indent=2), encoding='utf-8')
BUILDING_B_JSON.write_text(json.dumps(building_b_payload, indent=2), encoding='utf-8')
REMAINING_JSON.write_text(json.dumps(remaining_positions, indent=2), encoding='utf-8')

BUILDING_A_JSON.name, BUILDING_B_JSON.name, REMAINING_JSON.name

## Next Grasshopper tools to implement

1. `import_building_boundary_04`: receive a Python-generated boundary and create Rhino geometry.
2. `remaining_buildable_positions_04`: pixelize the remaining site and send candidate positions back.
3. `requested_position_checker_04`: evaluate a user-requested point and return geometry facts plus alternatives.

Once those exist in Swiftlet, replace the local mock calls in this notebook with real MCP calls.